In [38]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import glob

sns.set_style('whitegrid')

import matplotlib
# set font size to 16
matplotlib.rcParams.update({'font.size': 16})

ROOT = "./CIC2019/"
SEED = 4

In [39]:
files_cic2019 = glob.glob(ROOT + "03-11/*.csv")
files_cic2019.sort()
files_cic2019

['./CIC2019/03-11/01_Portmap.csv',
 './CIC2019/03-11/03_LDAP.csv',
 './CIC2019/03-11/04_MSSQL.csv',
 './CIC2019/03-11/05_UDP.csv',
 './CIC2019/03-11/06_UDPLag.csv',
 './CIC2019/03-11/07_Syn.csv']

In [40]:
dfs = []
for task_id, file in enumerate(tqdm(files_cic2019)):
    df = pd.read_csv(file, engine='pyarrow')
    df.columns = df.columns.str.strip()
    benign_df = df[df['Label'] == 'BENIGN']
    if len(benign_df) > 5000:
        benign_df = benign_df.sample(n=5000, random_state=SEED)
    malicious_df = df[df['Label'] != 'BENIGN']
    malicous_sampled_df = malicious_df.sample(n=2*len(benign_df), random_state=SEED)
    undersampled_df = pd.concat([malicous_sampled_df, benign_df])
    undersampled_df["task_id"] = task_id
    dfs.append(undersampled_df)
df = pd.concat(dfs)

100%|██████████| 6/6 [00:26<00:00,  4.44s/it]


In [41]:
df_dropped = df.drop(columns=['Unnamed: 0', 'Flow ID', 'Source IP', 'Destination IP', 'Protocol', 'Source Port', 'Destination Port', 'Timestamp', 'SimillarHTTP', 'Inbound'])

In [42]:
# replace infinities with -1
df_dropped = df_dropped.replace([np.inf, -np.inf], -1)
# replace nans with -2
df_dropped = df_dropped.fillna(-2)

In [43]:
# define train set and test set for each task
X_train_tasks = []
X_test_tasks = []
y_train_tasks = []
y_test_tasks = []
for i in range(len(files_cic2019)):
    X_task = df_dropped[df_dropped["task_id"] == i].drop(["Label", "task_id"], axis=1).to_numpy()
    y_task = df_dropped[df_dropped["task_id"] == i]["Label"].to_numpy()
    X_train, X_test, y_train, y_test = train_test_split(X_task, y_task, test_size=0.2, random_state=SEED, stratify=y_task)
    X_train_tasks.append(X_train)
    X_test_tasks.append(X_test)
    y_train_tasks.append(y_train)
    y_test_tasks.append(y_test)

In [44]:
full_task_train_data = []
full_task_test_data = []
full_task_train_labels = []
full_task_test_labels = []
for task_id in range(len(X_train_tasks)):
    X_train_task, y_train_task = X_train_tasks[task_id], y_train_tasks[task_id]
    X_test_task, y_test_task = X_test_tasks[task_id], y_test_tasks[task_id]
    full_task_train_data.append(X_train_task)
    full_task_test_data.append(X_test_task)
    full_task_train_labels.append(y_train_task)
    full_task_test_labels.append(y_test_task)
full_model = RandomForestClassifier(n_estimators=10, n_jobs=-1)
full_model.fit(np.concatenate(full_task_train_data), np.concatenate(full_task_train_labels))
model_preds = full_model.predict(np.concatenate(full_task_test_data))
print(accuracy_score(np.concatenate(full_task_test_labels), model_preds))

0.9832884097035041


In [8]:
model = RandomForestClassifier(n_estimators=10, n_jobs=-1)
avg_task_f1_scores = []
task_f1_scores = []
dataset_size = []
X_train_al, _, y_train_al, _ = train_test_split(X_train_tasks[0], y_train_tasks[0], test_size=0.99, random_state=SEED, stratify=y_train_tasks[0])
al_dataset_size = len(X_train_al)
model.fit(X_train_al, y_train_al)
X_train_al = list(X_train_al)
y_train_al = list(y_train_al)
for task_id in range(len(X_train_tasks)):
    X_train_task, y_train_task = X_train_tasks[task_id], y_train_tasks[task_id]

    # learn on current task
    for i in range(len(X_train_task)):
        probas = model.predict_proba(X_train_task[i].reshape(1, -1))
        # Uncertainty sampling
        if np.max(probas) < 0.9:
            X_train_al.append(X_train_task[i])
            y_train_al.append(y_train_task[i])
            model.fit(X_train_al, y_train_al)
            al_dataset_size += 1
    # compute average f1-score on current task and past tasks
    f1_scores = []
    for past_id in range(task_id + 1):
        X_test_task, y_test_task = X_test_tasks[past_id], y_test_tasks[past_id]
        y_pred = model.predict(X_test_task)
        f1_scores.append(accuracy_score(y_test_task, y_pred))
    print(f"{f1_scores[-1]}, {al_dataset_size}")
    task_f1_scores.append(f1_scores[-1])
    avg_task_f1_scores.append(np.mean(f1_scores))
    dataset_size.append(al_dataset_size)

0.9975360788454769, 171
0.9726666666666667, 351
0.9874776386404294, 517
0.9957469431153642, 676
0.9983613273248668, 832
0.998, 878


In [9]:
print(avg_task_f1_scores)
print(np.array(dataset_size) * 100 / len(df))

[0.9975360788454769, 0.9182235128475889, 0.9276772229704869, 0.9349213546896405, 0.9444859005260244, 0.9568718533109329]
[0.23048928 0.47310958 0.69685942 0.91117401 1.12144494 1.1834479 ]
